## TB granuloma evolution -- baseline model
    Marta Llabrés Tomás-Valiente

### Comments
- This baseline model aims to reproduce the spatio-temporal dynamics of a single TB granuloma. 
- This model is based on:
    - **Lesion**: evolution through a diffusion-reaction equation
    - **Fibroblasts**: evolution through both a chemokinetic and chemotactic term, and a proliferation term. The first one is proportional to the gradient of the lesion and is what drives the speed of the fibroblasts. The second one drives the direction of the movement (in principle) to tighten the encapsulation ring. The third one is to model the action of the TGF-$\beta$, and it is proportional to the gradient of the lesion.
    - **Calcification**: the necrosis of the lesion starts after encapsulation is completed, and no more oxygen nor nutrients can access within. The idea is that necrosis starts at the core of the lesion and expands towards the outsides. We have modelled it with an initial seed at the centre and then a reaction-diffusion equation. Keep in mind that the diffusion term does not model a *real* diffusion, but mimicks the growth of necrosis.

  The PDE system looks like:

  $$
\partial_t c_l = k_l g c_l (c_l-a) + \nabla \left( D_l g \nabla c_l \right) - k_{cal} c_{cal} c_l
$$

$$
\partial_t c_f = k_f g |\nabla c_l| c_f + \nabla \left ( \chi_{kin} g |\nabla c_l| \nabla c_f \right ) - \nabla \left ( \chi_{tax} g  c_f \nabla c_l \right )
$$

$$
\partial_t c_{cal} = k_{cal} c_{cal} c_l + \nabla \left( D_{cal}  \nabla c_{cal} \right)
$$


-  The set of parameters is chosen so that a lesion with a radius of 1 mm is encapsulated approximately at day 28 post infection. 

### Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import imageio.v2 as imageio
import os
import cv2
import csv
from scipy.stats import qmc
from shapely.geometry import Polygon, Point, box
from shapely import wkt, affinity
from typing import List, Tuple, Dict
from scipy.optimize import minimize
import statistics as stat
import matplotlib as mpl
from matplotlib.patches import Rectangle
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec

# ------- let's import our evolution function ----
import baseline_model as mfun

### Parameters

In [ ]:
# fixed parameters
dx = 0.05 # (mm)
lx = 80 
ly = 120
dt = 0.00025 # (days)


# lesion
a = 0.14 # threshold for lesion growth (g/mm2)
eps = 1 # lesion sharpness
r0 = 1.5 # lesion initial radius (grid cells)
alfa = 1 # initial concentration of lesion, between [0,1]
count = 1 # number of lesions --ID identification--
age1 = 0.0 # (days)
k = 9  #lesion reaction coefficient (days^-1) 
D = 5e-3 # lesion diffusivity (mm^2/days)

# fibroblasts 
k_f = 7.4  # fibroblasts proliferation coefficient (days^-1)
xi_kin = 9e-2  # chemokinetic coefficient (mm^3/day)
xi_tax = 9e-2  # chemotactic coefficient (mm^2/day)

# calcification
k_cal = 5 # calcification reaction coefficient (days^-1)
D_cal = 3e-3 # calcification diffusion coefficient (mm^2/day)

### Geometric initialization

In [ ]:
# let's define our secondary lobule. 
    # In this case it is a smaller section of a lobule, with fibroblasts at the periphreral grids (which represent the interlobular septa). 

# regular grid creation
coord = ((0., 0.), (0., ly), (lx, ly), (lx,0.), (0., 0.)) 
central_region = Polygon(coord)
x, y = central_region.exterior.xy 

# lesion origin creation
origin1 = (21, 60) # we define the lesion origin there so that the distance to the septa is of roughly 1 mm. 

In [ ]:
# plot of the regular domain
x_mm, y_mm = [dx * np.array(coord) for coord in central_region.exterior.xy]

fig, ax = plt.subplots(figsize=(3, 4))
ax.plot(x_mm, y_mm, 'k-', linewidth=2)
ax.fill(x_mm, y_mm, color='darkgray', alpha=0.5)
ax.set_xlabel(r'$x$ (mm)')
ax.set_ylabel(r'$y$ (mm)')
ax.set_xlim(-0.2, lx*dx + 0.2)
ax.set_ylim(-0.2, ly*dx + 0.2)
ax.set_aspect('equal')
plt.tight_layout()
#plt.savefig("regular_area.pdf", dpi=300, bbox_inches='tight')
plt.show()

### Simulation

In [ ]:
# we create a dictionary with the parameters
fixed_params = {"a": a, "eps": eps, "r0":r0, "alfa": alfa, "count": count, "dx": dx, "lx": lx, "ly": ly, "dt": dt, "age1": age1, "central_region": central_region, "origin1": origin1} 
estimated_params = {"k": k, "D": D, "xi": xi_kin, "xi_tax": xi_tax, "k_f": k_f, "k_cal": k_cal, "D_cal": D_cal}
params = {**estimated_params, **fixed_params}

print(params.keys())

In [ ]:
###### let's run the simulation
iter = 200000
state, ages, origins, radius, radius_c, full_radius, calcification, c_f1, max_grad, DIF_kin_vector, DIF_tax_vector, prol_vector, cal_dif_vec, phases, cfl_cond, overshoot_vec, n_cells_corrected_vec, c_l_time, c_cal_time, c_f_time = mfun.model_evolution_loop(iter, params)

### Results analysis and plots

In [ ]:
# heatmaps for the variables
cmap_cl = plt.cm.YlOrRd     
cmap_cf = plt.cm.YlGnBu     
cmap_ccal = LinearSegmentedColormap.from_list('ccal', ['#ffffe5', '#737373', '#000000'])

mpl.rcParams.update({"text.usetex": True, "font.family": "serif", "font.serif": ["Computer Modern Roman"], "axes.labelsize": 12, "font.size": 13, "legend.fontsize": 11, "xtick.labelsize": 11, "ytick.labelsize": 11})

# phases visualization
# --- colour map for each phase ---
phase_colors = {0:'#ffffff', 1:'#fff3cd', 2:'#ffd6a5', 3:'#cce5ff', 4:'#d4edda'}
phase_labels = {0: 'Nucleation', 1: 'Phase I: Growing', 2: 'Phase II: Encapsulating', 3: 'Phase III: Calcifying', 4: 'Phase IV: Resolved'}

In [ ]:
# data organization
c_l = state["c_l_1"].T 
c_f = c_f1.T
c_cal = calcification["c_cal_1"].T

# radius evolution variables
c_tot = c_l + c_f + c_cal
rad_les = radius["r_1"]
rad_calc = radius_c["r_c_1"]
full_radius = full_radius["full_r_1"]

time = np.arange(iter) * dt
extent_full = [0, c_l.shape[1]*dx, 0, c_l.shape[0]*dx]


In [ ]:
df = pd.DataFrame({"rad_les": rad_les, "rad_calc": rad_calc, "full_radius": full_radius})
#df.to_csv(f"radius_baseline.csv", index=False)

##### initialisation plots

In [ ]:
# Variables at initial iteration (zoomed vision of lesion)

# zoom window 
zoom_half = 20
i0, j0 = origins["origin_1"]
i_min, i_max = j0 - zoom_half, j0 + zoom_half
j_min, j_max = i0 - zoom_half, i0 + zoom_half
c_l_zoom = c_l[i_min:i_max, j_min:j_max]
extent_zoom = [j_min*dx, j_max*dx, i_min*dx, i_max*dx]

fig, axes = plt.subplots(1, 3, figsize=(12, 5))
# 1) lesion density map
ax_cl = axes[0]
im_cl = ax_cl.imshow(c_l_zoom, cmap=cmap_cl, origin='lower', extent=extent_zoom, interpolation='bilinear', vmin=0, vmax=1)
ax_cl.set_xlabel(r'$x$ (mm)', fontsize = '17')
ax_cl.set_ylabel(r'$y$ (mm)', fontsize = '17')
divider = make_axes_locatable(ax_cl)
cax_cl = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im_cl, cax=cax_cl, label=r'$c_l$')

# inset in the lesion figure
ax_inset = ax_cl.inset_axes([0.65, 0.59, 0.35, 0.35])
ax_inset.imshow(c_l, cmap=cmap_cl, origin='lower', extent=extent_full, interpolation='bilinear', vmin=0, vmax=1)
rect = Rectangle((j_min*dx, i_min*dx), (j_max - j_min)*dx, (i_max - i_min)*dx, edgecolor='darkgoldenrod', facecolor='none', linewidth=1.2)
ax_inset.add_patch(rect)
ax_inset.set_xticks([])
ax_inset.set_yticks([])
ax_inset.set_title(r'Full domain', fontsize= 10)

# fibroblasts density map
ax_cf = axes[1]
im_cf = ax_cf.imshow(c_f, cmap=cmap_cf, origin='lower', extent=extent_full, interpolation='bilinear', vmin=0, vmax=1)
ax_cf.set_xlabel(r'$x$ (mm)', fontsize = '17')
ax_cf.set_ylabel(r'$y$ (mm)', fontsize = '17')
divider2 = make_axes_locatable(ax_cf)
cax_cf = divider2.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im_cf, cax=cax_cf, label=r'$c_f$')


# calcification density map
ax_ccal = axes[2]
im_ccal = ax_ccal.imshow(c_cal, cmap=cmap_ccal, origin='lower', extent=extent_full, interpolation='bilinear', vmin=0, vmax=1)
ax_ccal.set_xlabel(r'$x$ (mm)', fontsize = '17')
ax_ccal.set_ylabel(r'$y$ (mm)', fontsize = '17')
divider2 = make_axes_locatable(ax_ccal)
cax_ccal = divider2.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im_ccal, cax=cax_ccal, label=r'$c_{cal}$')

plt.tight_layout()
plt.savefig("init_conditions.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# initial lesion profile for different eps values
r = np.linspace(0, 0.5, 500)

epsilons = [0.5, 1.0, 2.0]  # in grid cells * dx
labels = [r'$\varepsilon = 0.5$', r'$\varepsilon = 1.0$', r'$\varepsilon = 2.0$']
colors = ['steelblue', 'darkorange', 'seagreen']

fig, ax = plt.subplots(figsize=(6, 4))

for eps, label, color in zip(epsilons, labels, colors):
    eps_mm = eps * 0.05
    c_l = (alfa / 2) * (1 - np.tanh((r - r0*dx) / eps_mm))
    ax.plot(r, c_l, label=label, color=color, linewidth=2)

ax.axvline(x=r_tanh, color='gray', linestyle='--', linewidth=1)
ax.axhline(y=a, color='red', linestyle=':', linewidth=1.5)

ax.annotate(r'$r_{tanh}$', xy=(r_tanh, 0.02), xytext=(r_tanh + 0.02, 0.08), color='gray', fontsize=10)
ax.annotate(r'$a$', xy=(a, 0.02), xytext=(a + 0.02, 0.08), color='red', fontsize=10)

ax.set_xlabel(r'$|\mathbf{x} - \mathbf{x}_0|$ (mm)', fontsize=12)
ax.set_ylabel(r'$c_l(\mathbf{x}, 0)$', fontsize=12)
ax.set_xlim(0, 0.5)
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
#plt.savefig('lesion_init_profile.png', dpi=300, bbox_inches='tight')
plt.show()

##### final variable plots

In [ ]:
print(f'The lesion radius at day 28 was: {radius['r_1'][int(28/dt)-1]}')
r_crown = full_radius[int(28/dt)-1] - rad_les[int(28/dt)-1]
print(f'The fibroblasts crown thickness at day 28 was: {r_crown}')

In [ ]:
fig = plt.figure(figsize=(6, 10))
outer = gridspec.GridSpec(3, 1, figure=fig, hspace=0.3)

rows = [([c_l_time['c_l_79999'].T,   c_l_time['c_l_103999'].T,   c_l_time['c_l_149999'].T], cmap_cl,  r'$c_l$'),
    ([c_f_time['c_f_79999'].T,   c_f_time['c_f_103999'].T,   c_f_time['c_f_149999'].T],  cmap_cf,   r'$c_f$'),
    ([c_cal_time['c_cal_79999'].T,   c_cal_time['c_cal_103999'].T,   c_cal_time['c_cal_149999'].T], cmap_ccal, r'$c_{cal}$')]
    # day 20                      # day 26                        # day 40
letters = ['(a)', '(b)', '(c)']
time_labels = ['Day 20', 'Day 26', 'Day 40']

for i, (letter, (data_list, cmap, clabel)) in enumerate(zip(letters, rows)):
    inner = gridspec.GridSpecFromSubplotSpec(
        1, 4,
        subplot_spec=outer[i],
        wspace=0.05,
        width_ratios=[1, 1, 1, 0.05] 
    )

    for j, data in enumerate(data_list):
        ax = fig.add_subplot(inner[j])
        im = ax.imshow(data, cmap=cmap, origin='lower', extent=extent_full,
                       interpolation='bilinear', vmin=0, vmax=1)
        ax.set_xlabel(r'$x$ (mm)', fontsize = '14')

        if j == 0:
            ax.set_ylabel(r'$y$ (mm)', fontsize = '14')
            ax.set_title(letter, loc='left', fontweight='bold', fontsize = '17')
        else:
            ax.tick_params(labelleft=False)

        if i == 0:
            ax.set_title(time_labels[j], fontsize=15)

    cax = fig.add_subplot(inner[3])
    plt.colorbar(im, cax=cax, label=clabel)

# plt.savefig("baseline_outputs.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# variables at final iteration
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1) lesion density
ax_cl = axes[0]
im_cl = ax_cl.imshow(c_l, cmap=cmap_cl, origin='lower', extent=extent_full, interpolation='bilinear', vmin = 0, vmax = 1)
ax_cl.set_xlabel(r'$x$ (mm)')
ax_cl.set_ylabel(r'$y$ (mm)')
#ax_cl.set_title(r'(a)', loc='left')
divider = make_axes_locatable(ax_cl)
cax_cl = divider.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im_cl, cax=cax_cl, label=r'$c_l$')

# 2) fibroblast density
ax_cf = axes[1]
im_cf = ax_cf.imshow(c_f, cmap=cmap_cf, origin='lower', extent=extent_full, interpolation='bilinear', vmin = 0, vmax = 1)
ax_cf.set_xlabel(r'$x$ (mm)')
ax_cf.set_ylabel(r'$y$ (mm)')
#ax_cf.set_title(r'(b)', loc='left')
divider2 = make_axes_locatable(ax_cf)
cax_cf = divider2.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im_cf, cax=cax_cf, label=r'$c_f$')

# 3) calcification density
ax_ccal = axes[2]
cal_max = np.max(c_cal) if np.max(c_cal) > 1e-6 else 1.0  # fallback if empty
im_ccal = ax_ccal.imshow(c_cal, cmap=cmap_ccal, origin='lower', extent=extent_full, interpolation='bilinear', vmin=0, vmax=1)
ax_ccal.set_xlabel(r'$x$ (mm)')
ax_ccal.set_ylabel(r'$y$ (mm)')
#ax_ccal.set_title(r'(c)', loc='left')
divider3 = make_axes_locatable(ax_ccal)
cax_ccal = divider3.append_axes("right", size="5%", pad=0.05)
plt.colorbar(im_ccal, cax=cax_ccal, label=r'$c_{cal}$')

plt.tight_layout()
# plt.savefig("final_iteration.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# lesion radius evolution and phases in background
fig, ax = plt.subplots(figsize=(10, 4))

e = 1 # lesion ID
phase_array = phases[f'phase_{e}']
current_phase = phase_array[0]
start_idx = 0

for idx in range(1, iter):
    if phase_array[idx] != current_phase or idx == iter - 1:
        ax.axvspan(time[start_idx], time[idx], color=phase_colors.get(int(current_phase), 'white'), alpha=1.0, lw=0)
        start_idx = idx
        current_phase = phase_array[idx]

ax.plot(time, rad_les, linewidth=1.0, color='#bd0026', label=r'$r_{lesion}(t)$')
ax.plot(time, rad_calc, linewidth=1.0, color='#495057', label=r'$r_{cal}(t)$')
ax.plot(time, full_radius, linewidth=1.0, color='#005f73', label=r'$r_{granuloma}(t)$')

phase_patches = [Patch(facecolor=phase_colors[p], alpha=1.0, label=phase_labels[p]) for p in sorted(phase_colors.keys()) if p in np.unique(phases['phase_1'])]

radius_handles, _ = ax.get_legend_handles_labels()
ax.legend(handles=radius_handles + phase_patches, loc='upper left', fontsize=12)



ax.set_xlabel(r'$t$ (days)', fontsize = '17')
ax.set_ylabel(r'$r(t)$ (mm)', fontsize = '17')
ax.set_xlim(time[0], time[-1])
ax.set_ylim(bottom=0)
ax.grid(True, alpha=0.3)

plt.tight_layout()
# plt.savefig('baseline_radius_phases.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# plot of the fibroblasts term contribution
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(time, DIF_kin_vector, label=r'Chemokinesis $\chi_{chem}$', color='#4299f5', linewidth=1.5)
ax.plot(time, DIF_tax_vector, label=r'Chemotaxis $\chi_{tax}$', color='#f5a142', linewidth=1.5)
ax.plot(time, prol_vector, label=r'Proliferation $k_f$', color='#42c982', linewidth=1.5)


e = 1 # lesion ID
phase_array = phases[f'phase_{e}']
current_phase = phase_array[0]
start_idx = 0

for idx in range(1, iter):
    if phase_array[idx] != current_phase or idx == iter - 1:
        ax.axvspan(time[start_idx], time[idx], color=phase_colors.get(int(current_phase), 'white'), alpha=1.0, lw=0)
        start_idx = idx  
        current_phase = phase_array[idx]



ax.set_xlabel(r'$t$ (days)', fontsize = '17')
ax.set_ylabel(r'Term magnitude', fontsize = '17')
ax.legend(fontsize=12)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.grid(True, alpha=0.3)

phase_patches = [Patch(facecolor=phase_colors[p], alpha=1.0, label=phase_labels[p]) for p in sorted(phase_colors.keys()) if p in np.unique(phases['phase_1'])]

plt.tight_layout()
# plt.savefig('fibroblast_terms.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
time_days = np.arange(len(cfl_cond)) * dt

plt.figure(figsize=(6,4))
plt.plot(time_days, cfl_cond, lw=1.2, label='CFL', color = '#851613')

plt.axhline(1, linestyle='--', color='k', lw = 0.8, label='Stability threshold')

plt.xlabel('Time (days)', fontsize = '17')
plt.ylabel(r'$4D_{\max}^{\rm eff}\Delta t/\Delta x^2$', fontsize = '17')
plt.legend(fontsize = '12')
plt.tight_layout()
plt.grid(True, alpha=0.3)
plt.xlim(time[0], time[-1])

# plt.savefig('cfl_cond.pdf', dpi=300, bbox_inches='tight')
plt.show()